# 02 · Models as Tools — data, GLM, XGBoost, SHAP

**Agentic AI for Actuaries** · IFoA Workshop · 15 May 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 1, Parts 3–4. 
**You will:** inspect a realistic motor book, find a leakage trap, fit a Poisson GLM and an XGBoost challenger, referee them with an out-of-time lift chart, explain them with SHAP, run a fairness spot-check — and then wrap it all as **agent-ready tools** (the exact functions the capstone agent calls in notebook 05).

In [ ]:
%pip install -q xgboost shap statsmodels scikit-learn

## §1 · Generate and inspect ABC Motor 2024
Self-contained synthetic data — the notebook runs anywhere, no download. Then the three commands you run on **every** dataset before anything else.

In [ ]:
# --- ABC Motor 2024: synthetic hypothetical dataset (self-contained, no download needed) ---
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 50_000

motor = pd.DataFrame({
    "policy_id": [f"ABC-MOT-{i:06d}" for i in range(1, N + 1)],
    "vehicle_age_years": rng.integers(0, 16, N),
    "vehicle_make": rng.choice(["Maruti", "Hyundai", "Tata", "Mahindra", "Honda"], N,
                               p=[0.35, 0.25, 0.18, 0.12, 0.10]),
    "vehicle_segment": rng.choice(["Hatchback", "Sedan", "SUV", "MUV"], N,
                                  p=[0.45, 0.25, 0.22, 0.08]),
    "cubic_capacity": rng.choice([998, 1197, 1497, 1997, 2179], N),
    "ncb_pct": rng.choice([0, 20, 25, 35, 45, 50], N, p=[0.30, 0.15, 0.12, 0.15, 0.10, 0.18]),
    "policyholder_age": rng.integers(19, 75, N),
    "policyholder_gender": rng.choice(["M", "F"], N, p=[0.72, 0.28]),
    "region": rng.choice(["Tier1", "Tier2", "Tier3"], N, p=[0.40, 0.35, 0.25]),
    "prior_claims_3y": rng.choice([0, 1, 2, 3], N, p=[0.70, 0.20, 0.07, 0.03]),
})
motor["idv_inr"] = (900_000 * 0.9 ** motor["vehicle_age_years"]
                    * rng.uniform(0.8, 1.2, N)).round(-3)
# earned exposure over each policy's own observation year (mid-term entries/exits)
motor["exposure_years"] = rng.uniform(0.25, 1.0, N).round(3)
# underwriting cohort month — used for the out-of-time split (each cohort observed over its full policy year)
motor["inception_month"] = rng.integers(1, 13, N)

# True frequency model (the "world"): base 8% with realistic loadings
lin = (np.log(0.062)
       + 0.045 * motor["vehicle_age_years"]
       - 0.009 * motor["ncb_pct"]
       + 0.20 * motor["prior_claims_3y"]
       + np.where(motor["region"] == "Tier1", 0.12, np.where(motor["region"] == "Tier3", -0.10, 0.0))
       + np.where(motor["vehicle_segment"] == "SUV", 0.10, 0.0))
motor["claim_count"] = rng.poisson(np.exp(lin) * motor["exposure_years"])
# severity: Gamma, mean ~38k, only where claims exist
sev = rng.gamma(shape=2.0, scale=19_000, size=N)
motor["claim_amount_inr"] = (motor["claim_count"] * sev).round(0)

print("Shape:", motor.shape)
freq = motor.claim_count.sum() / motor.exposure_years.sum()
sev_mean = motor.loc[motor.claim_count > 0, "claim_amount_inr"].sum() / max(motor.claim_count.sum(), 1)
print(f"Portfolio frequency: {freq:.3f} per policy-year | mean severity: INR {sev_mean:,.0f}")
motor.head()

In [ ]:
motor.info()
motor.describe().T.head(12)

### Exposure-weighted frequency by vehicle age
Sum claims and exposure **separately**, then divide — never `groupby().mean()` on partial-year policies.

In [ ]:
bins, labels = [-0.5, 2, 5, 10, 20], ["0-2", "3-5", "6-10", "11+"]
motor["age_band"] = pd.cut(motor.vehicle_age_years, bins=bins, labels=labels)
freq_by_band = (motor.groupby("age_band", observed=True)
                     .apply(lambda g: g.claim_count.sum() / g.exposure_years.sum(),
                            include_groups=False))
ax = freq_by_band.plot(kind="bar", title="Exposure-weighted frequency by vehicle age band")
ax.set_ylabel("claims per policy-year");

## §2 · EXERCISE — find the column that breaks the model (3 minutes)
We predict whether a policy claimed at all. The quick model below scores a **perfect AUC**. Nothing is that good. One column gives the answer away — find it. *(Hint: it's measured in rupees.)*

**Why this matters for agents:** a human might smell a perfect AUC; an agent will happily fit, report and ship it — unless the tool it calls refuses leaky features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y_bin = (motor.claim_count > 0).astype(int)
X_leaky = motor.select_dtypes("number").drop(columns=["claim_count"])   # numeric kitchen sink
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y_bin, random_state=0)
auc = roc_auc_score(yte, LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
print(f"Test AUC = {auc:.3f}   <-- alarm bells")
print("\nColumns offered to the model:", list(X_leaky.columns))

In [ ]:
# REVEAL — run after you've guessed
# claim_amount_inr is non-zero exactly when the target is 1: pure target leakage.
# Rule: if a feature is knowable only AFTER the event you're predicting, it's leakage. Drop it.
FEATURES = ["vehicle_age_years", "cubic_capacity", "ncb_pct",
            "policyholder_age", "prior_claims_3y"]          # numeric, lawful, pre-event
CATEGORICAL = ["vehicle_segment", "region"]
# Note: idv_inr is deliberately EXCLUDED — it is a deterministic function of vehicle age
# (multicollinearity would smear the age signal across two columns). Exercise: add it back
# to the GLM and watch the vehicle_age coefficient lose significance.
print("Governed feature list (lives INSIDE the tools from here on):", FEATURES + CATEGORICAL)

## §3 · Tool #1 — the Poisson GLM (frequency)
Same GLM you know from SAS/R: log link, `log(earned exposure)` offset, coefficients = rating relativities.

In [ ]:
import statsmodels.api as sm

def design_matrix(df):
    X = pd.get_dummies(df[FEATURES + CATEGORICAL], drop_first=True).astype(float)
    return sm.add_constant(X, has_constant="add")

X = design_matrix(motor)
y = motor["claim_count"]
offset = np.log(motor["exposure_years"])

glm_freq = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
print(glm_freq.summary().tables[1])

## §4 · Tool #2 — the XGBoost challenger + out-of-time split
Actuaries split by **time**, never at random: train on months 1–6, validate 7–9, test 10–12. Note `objective='count:poisson'` — we model counts, not squared error.

In [ ]:
from xgboost import XGBRegressor

train = motor[motor.inception_month <= 6]
val   = motor[motor.inception_month.between(7, 9)]
test  = motor[motor.inception_month >= 10]

def xy(df):
    X = pd.get_dummies(df[FEATURES + CATEGORICAL], drop_first=True).astype(float)
    return X, df.claim_count, df.exposure_years

Xtr, ytr, etr = xy(train); Xva, yva, eva = xy(val); Xte, yte, ete = xy(test)
Xva = Xva.reindex(columns=Xtr.columns, fill_value=0)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

xgb_freq = XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                        reg_lambda=1.0, objective="count:poisson", random_state=42)
# Train on FREQUENCY (count / earned exposure), weighted by exposure —
# the tree-model equivalent of the GLM's log-exposure offset.
xgb_freq.fit(Xtr, ytr / etr, sample_weight=etr, verbose=False)

glm_oot = sm.GLM(ytr, sm.add_constant(Xtr, has_constant="add"),
                 family=sm.families.Poisson(), offset=np.log(etr)).fit()
print("Both models fitted on H1, ready to score Q4 (never seen).")

## §5 · The referee — exposure-weighted lift table on Q4

In [ ]:
def lift_table(y_actual, y_pred, exposure, n=5):
    # quintiles on a 5,000-policy book; use deciles (n=10) on larger books
    df = pd.DataFrame({"pred": y_pred, "actual": y_actual, "expo": exposure})
    df["band"] = pd.qcut(df.pred.rank(method="first"), n, labels=False) + 1
    return (df.groupby("band")
              .apply(lambda g: g.actual.sum() / g.expo.sum(), include_groups=False)
              .rename("observed_freq"))

glm_pred = glm_oot.predict(sm.add_constant(Xte, has_constant="add"), offset=np.log(ete))
xgb_pred = xgb_freq.predict(Xte) * ete   # model predicts a rate; x exposure = expected counts

lift = pd.DataFrame({"GLM": lift_table(yte, glm_pred, ete),
                     "XGBoost": lift_table(yte, xgb_pred, ete)})
print(lift.round(4))
lift.plot(marker="o", title="Out-of-time lift — observed frequency by predicted quintile (Q4 cohort)")
print(f"\nTop/bottom band ratio — GLM: {lift.GLM.iloc[-1]/lift.GLM.iloc[0]:.1f}x | "
      f"XGB: {lift.XGBoost.iloc[-1]/lift.XGBoost.iloc[0]:.1f}x")

**Read it like a pricing committee:** on a 5,000-policy book the GLM holds its own on calibration; the tree usually wins segmentation at the extremes — where pricing leakage hides. They answer different questions, which is why the agent gets *both*.

## §6 · Tool #3 — SHAP (global + local)

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_freq)
sv = explainer(Xte)
shap.summary_plot(sv, Xte, max_display=8)     # GLOBAL: has the tree learned anything perverse?

In [ ]:
i = 0                                          # LOCAL: why did THIS policy get that score?
shap.plots.waterfall(sv[i], max_display=8)
print("Every push traces to a data column — this is the customer letter and the regulator's answer.")

## §7 · Fairness spot-check — calibration within subgroup
Predicted vs observed by gender × age band, **even though gender is not a model feature** — that's the proxy-discrimination test from the slides.

In [ ]:
audit = test.copy()
audit["pred_count"] = xgb_pred
audit["ph_band"] = pd.cut(audit.policyholder_age, [18, 35, 60, 80])

def cell_rates(g):
    return pd.Series({"observed": g.claim_count.sum() / g.exposure_years.sum(),
                      "predicted": g.pred_count.sum() / g.exposure_years.sum()})

report = (audit.groupby(["policyholder_gender", "ph_band"], observed=True)
               .apply(cell_rates, include_groups=False).round(4))
print(report)
gap = (report.observed - report.predicted).abs().max()
print(f"\nLargest |observed - predicted| gap: {gap:.4f} "
      f"{'✓ within tolerance (0.03)' if gap < 0.03 else '⚠ investigate before shipping'}")
# Exposure-weighted within-cell calibration — the fairness definition we chose and can defend.

## §8 · Wrap it all as agent-ready tools
This is the whole point of the morning: every artefact above becomes a **typed function with a docstring** — the exact tools the capstone agent calls in notebook 05. Judgement (feature list, split policy, fairness tolerance) is encoded **once, inside the tool**.

In [ ]:
def fit_glm_tool() -> dict:
    """Fit the governed Poisson frequency GLM on ABC Motor 2024 (H1 train).
    Returns coefficient table and train window. Feature list is fixed inside the tool."""
    return {"model": "Poisson GLM", "train_window": "2024 months 1-6",
            "coefficients": glm_oot.params.round(4).to_dict()}

def fit_xgb_tool() -> dict:
    """Fit the governed XGBoost frequency challenger. Hyperparameters and features fixed inside."""
    return {"model": "XGBoost count:poisson", "n_estimators": 500, "max_depth": 4}

def lift_table_tool() -> dict:
    """Out-of-time (Q4 2024) decile lift for GLM vs XGBoost. The referee."""
    return lift.round(4).to_dict()

def shap_top_features_tool() -> dict:
    """Global mean |SHAP| ranking for the XGBoost model."""
    mean_abs = pd.Series(np.abs(sv.values).mean(axis=0), index=Xte.columns)
    return mean_abs.sort_values(ascending=False).head(6).round(4).to_dict()

print(lift_table_tool())
print(shap_top_features_tool())
print("\nFour governed tools, ready for an agent. See you in notebook 05.")